# Sizing de tokens e custo (whatsapp-finance-agent)

Este notebook calcula **limites recomendados** de tokens por iteração e por dia a partir de:

- um **alvo de gasto em USD por cliente por dia**;
- um **teto por bloco de iterações** (ex.: no máximo *X* USD a cada 10 chamadas ao modelo);
- preços por 1M tokens (entrada e saída) do modelo escolhido.

> **Nota sobre valores monetários:** `usd_max_per_block` pode ser **1,5 centavo de dólar** (`0.015`) ou **um dólar e cinquenta** (`1.50`) — ajuste o parâmetro na célula seguinte conforme a sua intenção.

Os valores sugeridos no final servem para preencher `LLM_MAX_OUTPUT_TOKENS`, `LLM_MAX_REQUEST_TOKENS` e `LLM_DAILY_TOKEN_BUDGET_PER_USER` no `.env` do bot.


In [ ]:
# --- Parâmetros (edite aqui) ---

# Alvo de receita/custo máximo por cliente e por dia (USD)
usd_target_per_client_per_day = 0.10

# Quantas chamadas ao modelo (iterações) por cliente por dia em média
iterations_per_day = 50

# Bloco de controle de custo: teto em USD a cada N iterações
iterations_per_cost_block = 10
# Ex.: 0.015 = 1,5 centavo de dólar por 10 iterações; use 1.50 se quiser um dólar e cinquenta
usd_max_per_block = 0.015

# Preços do modelo (USD por 1 milhão de tokens) — atualize com a tabela da OpenAI
usd_per_1m_input = 0.15
usd_per_1m_output = 0.60

# Fração dos tokens totais por iteração que são de entrada (prompt), entre 0 e 1.
# O restante é tratado como saída para o dimensionamento.
input_fraction_of_total = 0.72

# Margem de segurança sobre o custo (1.0 = nenhuma; 1.15 = 15% mais conservador)
safety_multiplier = 1.12


In [ ]:
# --- Cálculos ---

import math

f = min(max(input_fraction_of_total, 0.01), 0.99)
p_in = usd_per_1m_input / 1_000_000
p_out = usd_per_1m_output / 1_000_000

usd_per_iteration_from_block = usd_max_per_block / iterations_per_cost_block
usd_per_iteration_from_daily = (
    usd_target_per_client_per_day / iterations_per_day
    if iterations_per_day > 0
    else float("inf")
)

usd_cap_per_iteration = min(usd_per_iteration_from_block, usd_per_iteration_from_daily)
usd_cap_per_iteration /= safety_multiplier

# Custo por iteração se total_tokens = K: f*K p_in + (1-f)*K p_out
marginal_usd_per_token = f * p_in + (1 - f) * p_out
K_max = usd_cap_per_iteration / marginal_usd_per_token if marginal_usd_per_token > 0 else 0

est_input_tokens = f * K_max
est_output_tokens = (1 - f) * K_max

llm_max_output_tokens = max(1, math.floor(est_output_tokens))
llm_max_request_tokens = max(
    llm_max_output_tokens + 1, math.ceil(est_input_tokens + llm_max_output_tokens)
)
llm_daily_token_budget = max(1, math.floor(K_max * iterations_per_day))

print("=== Entradas resumidas ===")
print(f"Alvo USD/dia/cliente: {usd_target_per_client_per_day}")
print(f"Iterações/dia: {iterations_per_day}")
print(
    f"Teto USD / {iterations_per_cost_block} iterações: {usd_max_per_block}  ->  {usd_per_iteration_from_block:.6f} USD/iter"
)
print(f"Alvo implícito USD/iter (só diário): {usd_per_iteration_from_daily:.6f}")
print(f"USD/iter usado (após min e margem {safety_multiplier}): {usd_cap_per_iteration:.6f}")
print()
print("=== Tokens por iteração (modelo linear) ===")
print(f"Total tokens/iter (K): ~{K_max:.0f}")
print(f"~Entrada: {est_input_tokens:.0f}  ~Saída: {est_output_tokens:.0f}")
print()
print("=== Sugestão para .env ===")
print(f"LLM_MAX_OUTPUT_TOKENS={llm_max_output_tokens}")
print(f"LLM_MAX_REQUEST_TOKENS={llm_max_request_tokens}")
print(f"LLM_DAILY_TOKEN_BUDGET_PER_USER={llm_daily_token_budget}")
print()
print("Custo USD/iter estimado (com K acima):", K_max * marginal_usd_per_token)
print("Custo USD/dia estimado:", K_max * marginal_usd_per_token * iterations_per_day)
